# 🏀 NBA Shot Success — Exploration (EDA + Shot Chart)

Analyse exploratoire du dataset de tirs et visualisation d'une *shot chart*.

**Prérequis** : avoir lancé l'ingestion puis la construction des features :
```bash
python -m src.data_ingestion --seasons 2023-24 --max-teams 5
python -m src.features --seasons 2023-24
```

> ℹ️ Le dataset modèle et la *shot chart* viennent tous deux de `ShotChartDetail`
> (tir par tir, avec coordonnées `LOC_X` / `LOC_Y`). Les features défenseur / shot clock
> / dribbles ne sont plus disponibles par tir dans l'API publique NBA (cf. README).

In [ ]:
import sys
from pathlib import Path

# Permet d'importer le package `src` depuis le dossier notebooks/
sys.path.append(str(Path.cwd().parent))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

try:
    import seaborn as sns
    sns.set_theme(style="whitegrid")
except ImportError:
    pass

from src import config

%matplotlib inline

## 1. Chargement du dataset de features

In [ ]:
df = pd.read_parquet(config.PROCESSED_DATASET_PATH)
print(f"{len(df):,} tirs")
print(f"FG% global : {df['shot_made'].mean():.3f}")
df.head()

In [ ]:
df[config.NUMERIC_FEATURES + [config.TARGET_COLUMN]].describe()

## 2. Distribution des features

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
for ax, col in zip(axes.ravel(), config.NUMERIC_FEATURES):
    ax.hist(df[col].dropna(), bins=40, color="steelblue", alpha=0.8)
    ax.set_title(col)
plt.tight_layout()
plt.show()

## 3. Réussite (FG%) en fonction du contexte

On découpe chaque feature en *bins* et on regarde le taux de réussite moyen — c'est la
vraie question du projet : *qu'est-ce qui fait rentrer un tir ?*

In [ ]:
def fg_pct_by_bin(frame, col, bins):
    grp = frame.groupby(pd.cut(frame[col], bins))["shot_made"].agg(["mean", "count"])
    return grp

fig, axes = plt.subplots(2, 2, figsize=(14, 9))

fg_pct_by_bin(df, "shot_distance", np.arange(0, 35, 2))["mean"].plot(
    ax=axes[0, 0], marker="o", title="FG% vs distance du panier")
fg_pct_by_bin(df, "period_time_remaining", np.arange(0, 720, 60))["mean"].plot(
    ax=axes[0, 1], marker="o", color="green", title="FG% vs temps restant dans le quart (s)")
fg_pct_by_bin(df, "loc_y", np.arange(-50, 400, 25))["mean"].plot(
    ax=axes[1, 0], marker="o", color="darkorange", title="FG% vs position verticale (loc_y)")
df.groupby("shot_value")["shot_made"].mean().plot.bar(
    ax=axes[1, 1], color="purple", title="FG% par type de tir (2 vs 3 pts)")

for ax in axes.ravel():
    ax.set_ylabel("FG%")
plt.tight_layout()
plt.show()

In [ ]:
# Corrélations entre features numériques et la cible
corr = df[config.NUMERIC_FEATURES + ["shot_made"]].corr()
fig, ax = plt.subplots(figsize=(8, 6))
im = ax.imshow(corr, cmap="coolwarm", vmin=-1, vmax=1)
ax.set_xticks(range(len(corr))); ax.set_xticklabels(corr.columns, rotation=45, ha="right")
ax.set_yticks(range(len(corr))); ax.set_yticklabels(corr.columns)
for i in range(len(corr)):
    for j in range(len(corr)):
        ax.text(j, i, f"{corr.iloc[i, j]:.2f}", ha="center", va="center", fontsize=8)
fig.colorbar(im); plt.title("Matrice de corrélation"); plt.tight_layout(); plt.show()

## 4. Shot chart

`ShotChartDetail` fournit les coordonnées de terrain (`LOC_X`, `LOC_Y`). On récupère ici
les tirs d'un joueur donné pour tracer sa *shot chart* (réussis vs manqués, puis FG%
par zone en hexbin).

In [ ]:
from matplotlib.patches import Circle, Rectangle, Arc

def draw_court(ax=None, color="black", lw=2):
    """Dessine un demi-terrain NBA (coordonnées stats.nba : 1 unité = 0.1 pied)."""
    if ax is None:
        ax = plt.gca()
    hoop = Circle((0, 0), radius=7.5, linewidth=lw, color=color, fill=False)
    backboard = Rectangle((-30, -7.5), 60, -1, linewidth=lw, color=color)
    outer_box = Rectangle((-80, -47.5), 160, 190, linewidth=lw, color=color, fill=False)
    inner_box = Rectangle((-60, -47.5), 120, 190, linewidth=lw, color=color, fill=False)
    top_ft = Arc((0, 142.5), 120, 120, theta1=0, theta2=180, linewidth=lw, color=color)
    bottom_ft = Arc((0, 142.5), 120, 120, theta1=180, theta2=0, linewidth=lw,
                    color=color, linestyle="dashed")
    restricted = Arc((0, 0), 80, 80, theta1=0, theta2=180, linewidth=lw, color=color)
    corner_three_a = Rectangle((-220, -47.5), 0, 140, linewidth=lw, color=color)
    corner_three_b = Rectangle((220, -47.5), 0, 140, linewidth=lw, color=color)
    three_arc = Arc((0, 0), 475, 475, theta1=22, theta2=158, linewidth=lw, color=color)
    outer_lines = Rectangle((-250, -47.5), 500, 470, linewidth=lw, color=color, fill=False)
    for el in [hoop, backboard, outer_box, inner_box, top_ft, bottom_ft, restricted,
               corner_three_a, corner_three_b, three_arc, outer_lines]:
        ax.add_patch(el)
    return ax

In [ ]:
from nba_api.stats.endpoints import shotchartdetail

# Exemple : Stephen Curry (PLAYER_ID 201939), saison 2023-24.
PLAYER_ID = 201939
SEASON = "2023-24"

sc = shotchartdetail.ShotChartDetail(
    team_id=0,
    player_id=PLAYER_ID,
    season_nullable=SEASON,
    season_type_all_star="Regular Season",
    context_measure_simple="FGA",
    timeout=30,
)
shots = sc.get_data_frames()[0]
print(f"{len(shots)} tirs — FG% = {shots['SHOT_MADE_FLAG'].mean():.3f}")
shots[["LOC_X", "LOC_Y", "SHOT_DISTANCE", "SHOT_MADE_FLAG", "ACTION_TYPE"]].head()

In [ ]:
made = shots[shots["SHOT_MADE_FLAG"] == 1]
missed = shots[shots["SHOT_MADE_FLAG"] == 0]

fig, ax = plt.subplots(figsize=(9, 8.5))
ax.scatter(missed["LOC_X"], missed["LOC_Y"], c="red", marker="x", alpha=0.4, s=20,
           label="Manqué")
ax.scatter(made["LOC_X"], made["LOC_Y"], c="green", marker="o", alpha=0.5, s=20,
           label="Réussi")
draw_court(ax)
ax.set_xlim(-250, 250); ax.set_ylim(-50, 425)
ax.set_aspect("equal"); ax.axis("off")
ax.legend(loc="upper right")
ax.set_title(f"Shot chart — joueur {PLAYER_ID} ({SEASON})")
plt.show()

In [ ]:
# Variante : hexbin du FG% par zone (densité + efficacité)
fig, ax = plt.subplots(figsize=(9, 8.5))
hb = ax.hexbin(shots["LOC_X"], shots["LOC_Y"], C=shots["SHOT_MADE_FLAG"],
               gridsize=30, cmap="RdYlGn", extent=(-250, 250, -50, 425), mincnt=1)
draw_court(ax)
ax.set_xlim(-250, 250); ax.set_ylim(-50, 425)
ax.set_aspect("equal"); ax.axis("off")
fig.colorbar(hb, label="FG% par zone")
ax.set_title(f"FG% par zone — joueur {PLAYER_ID} ({SEASON})")
plt.show()

## 5. Pistes

- `shot_distance` et le type de tir (layup / Restricted Area vs jump shot) sont les
  leviers les plus nets — à confirmer dans `reports/shap_summary.png`.
- Idées de features additionnelles : interaction `shot_distance × shot_value`,
  angle du tir (`arctan2(loc_x, loc_y)`), indicateur *corner 3*, fin de quart-temps
  (`period_time_remaining < 3`), prise en compte du tireur (effet joueur).
- Limite connue : sans la distance du défenseur (retirée de l'API publique), le plafond
  de performance est ~0.65 d'AUC. Pour aller plus loin, il faudrait un dataset de
  tracking historique (p. ex. les *NBA shot logs 2014-15*).